# Force Prediction

Machine learning model and data pipeline for predicting profile at any given MVC level from motor neuron activations.

## Preprocessing

Sanitize and prepare all the data for the machine learning model. This is a crucial step in the machine learning pipeline. The quality of the data and the features used in the model will determine the model's performance.

### Conversion

Convert, process, and export processed original data to a more structured and useful format (pandas DataFrame).

In [108]:
import importlib
import sanitization.conversion

importlib.reload(sanitization.conversion)

from sanitization.conversion import Conversion

c = Conversion(prefix='sanitization', subjects=["thanasis"])

c.cleanup_data() # cleans the data directory and prepares for processing
c.process_subjects() # processes each subject and exports to txt files

Skipping thanasis as raw folder already exists

Processing data for thanasis

Skipping 10.1.mat as it has already been processed
Skipping 10.2.mat as it has already been processed
Skipping 10.3.mat as it has already been processed
Skipping 20.1.mat as it has already been processed
Skipping 20.2.mat as it has already been processed
Skipping 20.3.mat as it has already been processed
Skipping 40.1.mat as it has already been processed
Skipping 40.2.mat as it has already been processed
Skipping 40.3.mat as it has already been processed
Skipping 5.1.mat as it has already been processed
Skipping 5.2.mat as it has already been processed
Skipping 5.3.mat as it has already been processed
Skipping 60.1.mat as it has already been processed
Skipping 60.2.mat as it has already been processed
Skipping 60.3.mat as it has already been processed
Processing MVC.1.mat (MVC: 100, Trial: 1)...
Processing MVC.2.mat (MVC: 100, Trial: 2)...
Successfully processed data for thanasis


In [109]:
df = c.get_dataframe() # returns a pandas dataframe of the processed data
df

Loading data for thanasis...


,subject,mvc_level,trial_number,neuron_data,force_data
0,thanasis,5,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.012200930784608488, 0.0244018615692134..."
1,thanasis,5,2,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.0, 0.0, 0.0, -0.006100465392304244, 0...."
2,thanasis,5,3,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, -0.006100465392304244, -0.00610046539230..."
3,thanasis,10,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.0, -0.012200930784601383, 0.0, -0.0061..."
4,thanasis,10,2,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.006100465392304244, 0.0, 0.00610046539..."
5,thanasis,10,3,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, -0.006100465392300691, -0.00610046539230..."
6,thanasis,20,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.0, 0.006100465392304244, 0.0, 0.0, 0.0..."
7,thanasis,20,2,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.012200930784608488, 0.0061004653923042..."
8,thanasis,20,3,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.0, 0.01830139617690918, 0.006100465392..."
9,thanasis,40,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.0, 0.0, 0.0, -0.006100465392304244, 0...."


### Sanitization

Remove or correct any errors in the data. This includes removing outliers, filling in missing values, and correcting any other errors in the data. Some entries are not recoverable and will be "purged".

Below are the sanitization steps that will be taken:

#### Lack of neurons (`N`) — Purge
Number of recorded neurons (expect Max MVC) is below some acceptable constant value.

#### Spike (`S`)
An intense and major deviation from regular force trend.

#### Neuron Inconsistency (`NI`) — Purge
Major inconsistencies or large distances between activation times in vertically aligned neurons.

#### Measurement Decorrelation (`MD`) — Purge?
Discrepancies between measured neuron activations and force readings. An abundance of discrepancies (`MD`) in the dataset leads to General Measurement Decorrelation (`GMD`) and marshals a purge. 

#### Trend (`T`) — Purge
Force trend has a significant deviation from typical (average) force trend of it's peers. This feature ignores magnitude and strictly considers the relative shape in relation to others. 

More information and specific algorithm implementations can be found [here](https://github.com/paul-bokelman/force-prediction/issues/3).

In [112]:
import importlib
import sanitization.sanitization
import sanitization.constants

importlib.reload(sanitization.sanitization)
importlib.reload(sanitization.constants)

from sanitization.sanitization import Sanitization

sanitizer = Sanitization(df=df)
df = sanitizer.sanitize()

Purging entries with insufficient neuron data...
Purged 2 entries with insufficient neuron data. 15 entries remaining.


In [113]:
print(min(df[df["subject"] == "thanasis"]["force_data"].iloc[0]))
print(max(df[df["subject"] == "thanasis"]["force_data"].iloc[0]))
df[df["subject"] == "thanasis"]

-0.3599274581459255
1.1407870283608226


,subject,mvc_level,trial_number,neuron_data,force_data
0,thanasis,5,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.012200930784608488, 0.0244018615692134..."
1,thanasis,5,2,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.0, 0.0, 0.0, -0.006100465392304244, 0...."
2,thanasis,5,3,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, -0.006100465392304244, -0.00610046539230..."
3,thanasis,10,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.0, -0.012200930784601383, 0.0, -0.0061..."
4,thanasis,10,2,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.006100465392304244, 0.0, 0.00610046539..."
5,thanasis,10,3,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, -0.006100465392300691, -0.00610046539230..."
6,thanasis,20,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.0, 0.006100465392304244, 0.0, 0.0, 0.0..."
7,thanasis,20,2,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.012200930784608488, 0.0061004653923042..."
8,thanasis,20,3,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.0, 0.01830139617690918, 0.006100465392..."
9,thanasis,40,1,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[0.0, 0.0, 0.0, 0.0, -0.006100465392304244, 0...."
